# ***Both in One*** — **Image Analysis**

This notebook applies computer vision and image analysis techniques to study the visual evolution of the *Both in One* pastel pencil artworks across painting stages.

For each stage photograph, the following metrics are extracted:

- **Dominant colors** — K-means clustering in RGB space
- **Stroke density** — edge detection via the Canny method
- **Texture complexity** — Local Binary Patterns (LBP)
- **Fractal dimension** — box-counting method
- **Color harmony** — hue mean and standard deviation
- **Spatial painting density** — edge-based heatmaps

The extracted metrics are exported as a CSV file for subsequent statistical analysis in R.

---

**Author**: Julieta Arco  
**Project**: Both in One
**Medium**: Pastel pencil on Canson Mi-Teintes paper (50 × 65 cm)  
**Tools**: Python, OpenCV, scikit-learn, scikit-image, Matplotlib, Google Colab

---
## **Setup**

Install required libraries if running outside Google Colab or if import errors occur.

In [ ]:
# Uncomment if needed
# !pip install opencv-python-headless matplotlib scikit-image scikit-learn

---
## **1. Load Images**

Images are loaded from Google Drive in stage order. Each file corresponds to one painting session.

**Configuration**: Update `folder_path` and `image_filenames` to match your painting and folder structure before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
from PIL import Image

# --- Configuration ---
folder_path = "/content/drive/MyDrive/..../"  # Update as needed
painting_id = "BiO"                          # Painting identifier prefix

image_filenames = ['BiO1a.png', 'BiO2b.png', 'BiO2c.png', ...,
    # Add all stage filenames here in order
]
# ---------------------

images = []
loaded_stages = []

for filename in image_filenames:
    file_path = os.path.join(folder_path, filename)
    if os.path.exists(file_path):
        img = Image.open(file_path).convert("RGB")
        images.append(np.array(img))
        loaded_stages.append(filename)
        print(f"Loaded: {filename}")
    else:
        print(f"Not found: {filename}")

print(f"\nTotal stages loaded: {len(images)}")

---
## **2. Image Preparation**

To assess the effect of lighting normalization, two corrected versions of each image are generated and compared visually with the original:

- **CLAHE** (Contrast Limited Adaptive Histogram Equalization): enhances local contrast
- **Gamma correction**: adjusts overall brightness and tonal range

The original images are used for all subsequent metric extraction.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def adjust_gamma(image, gamma=1.8):
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(image, table)

clahe_imgs = [apply_clahe(img) for img in images]
gamma_imgs = [adjust_gamma(img, gamma=1.8) for img in images]

# Visual comparison
fig, axs = plt.subplots(len(images), 3, figsize=(12, len(images) * 3))

for i in range(len(images)):
    axs[i, 0].imshow(images[i])
    axs[i, 0].set_title(f"Stage {i+1} — Original")
    axs[i, 0].axis('off')

    axs[i, 1].imshow(clahe_imgs[i])
    axs[i, 1].set_title("CLAHE")
    axs[i, 1].axis('off')

    axs[i, 2].imshow(gamma_imgs[i])
    axs[i, 2].set_title("Gamma Corrected")
    axs[i, 2].axis('off')

plt.suptitle("Image Normalization Comparison", fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

---
## **3. Image Metrics**

### **3.1 Dominant Colors**

Five dominant colors are identified per stage using K-means clustering in RGB space. The relative size of each cluster estimates the proportional usage of each color in the image.

In [ ]:
from google.colab import files

# Create output folder
output_dir = "dominant_colors"
os.makedirs(output_dir, exist_ok=True)

for i, img in enumerate(images):
    colors, counts = extract_dominant_colors(img, n_colors=5)
    proportions = counts / counts.sum()

    fig, ax = plt.subplots(figsize=(6, 1))
    left = 0
    for color, prop in zip(colors, proportions):
        ax.barh(0, prop, left=left, color=color / 255)
        left += prop
    ax.set_title(f"Stage {i+1} — Dominant Colors (proportional)")
    ax.axis('off')
    plt.tight_layout()

    filepath = os.path.join(output_dir, f"dominant_colors_stage_{i+1:02d}.png")
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

# Zip and download to your computer
!zip -r dominant_colors.zip dominant_colors
files.download("dominant_colors.zip")

### **3.2 Stroke Density** — Edge Detection (Canny Method)

Edge detection identifies sharp transitions in pixel intensity, corresponding to stroke boundaries, contours, shading transitions, and pressure variations. Two metrics are computed:

- **Raw stroke density**: total number of edge pixels
- **Normalized stroke density (%)**: proportion of edge pixels relative to total image pixels

Higher values indicate richer texture and more detailed areas.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from google.colab import files

def detect_edges(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return cv2.Canny(gray, threshold1=50, threshold2=150)

def stroke_density_raw(edges):
    return int(np.sum(edges > 0))

def stroke_density_normalized(edges):
    return float(np.sum(edges > 0) / edges.size * 100)

# Create output folder
output_dir = "stroke_edges"
os.makedirs(output_dir, exist_ok=True)

stroke_densities_raw = []
stroke_densities_norm = []

for i, img in enumerate(images):
    edges = detect_edges(img)
    raw = stroke_density_raw(edges)
    norm = stroke_density_normalized(edges)
    stroke_densities_raw.append(raw)
    stroke_densities_norm.append(norm)

    plt.figure(figsize=(5, 5))
    plt.imshow(edges, cmap='Greys')
    plt.title(f"Stage {i+1} — Stroke Edges\nDensity: {norm:.2f}%")
    plt.axis('off')
    plt.tight_layout()

    # Save
    filepath = os.path.join(output_dir, f"stroke_edges_stage_{i+1:02d}.png")
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    print(f"Stage {i+1}: Raw = {raw} px | Normalized = {norm:.3f}%")

# Zip and download
!zip -r stroke_edges.zip stroke_edges
files.download("stroke_edges.zip")

### **3.3 Texture Complexity** — Local Binary Patterns (LBP)

LBP quantifies local texture by comparing each pixel's intensity to its neighbors. The standard deviation of the LBP map (LBP SD) serves as a proxy for overall texture complexity.

- **Higher LBP SD**: more texture diversity (varied stroke directions, strong contrast)
- **Lower LBP SD**: more uniform texture (soft shading, minimal marks)

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
import numpy as np

def texture_lbp_sd(image):
    gray = rgb2gray(image)
    gray_uint8 = (gray * 255).astype(np.uint8)
    lbp = local_binary_pattern(gray_uint8, P=8, R=1.0)
    return float(lbp.std())

texture_scores = []

for i, img in enumerate(images):
    score = texture_lbp_sd(img)
    texture_scores.append(score)
    print(f"Stage {i+1}: LBP SD = {score:.3f}")

### **3.4 Fractal Dimension** — Box-Counting Method

Fractal dimension quantifies the structural complexity and self-similarity of an image. The box-counting method covers the image with grids of decreasing size and counts how many boxes contain part of the structure.

- **~1.0**: simple, smooth structure — minimal marks
- **~1.5–1.7**: moderate detail — outlines, sparse texture
- **>1.8**: high complexity — dense strokes, fine detail, heavy shading

In [ ]:
import cv2
import numpy as np

def fractal_dimension(Z):
    assert len(Z.shape) == 2, "Input must be a 2D grayscale image"

    def boxcount(Z, k):
        S = np.add.reduceat(
            np.add.reduceat(Z, np.arange(0, Z.shape[0], k), axis=0),
            np.arange(0, Z.shape[1], k), axis=1)
        return len(np.where(S > 0)[0])

    Z = (Z < 128).astype(int)
    p = min(Z.shape)
    n = 2 ** np.floor(np.log2(p))
    sizes = 2 ** np.arange(int(np.log2(n)), 1, -1)
    counts = [boxcount(Z, int(s)) for s in sizes]
    coeffs = np.polyfit(np.log(sizes), np.log(counts), 1)
    return float(-coeffs[0])

fd_scores = []

for i, img in enumerate(images):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    fd = fractal_dimension(gray)
    fd_scores.append(fd)
    print(f"Stage {i+1}: Fractal Dimension = {fd:.3f}")

### **3.5 Color Harmony** — Hue Analysis

The hue component of the HSV color space is used to evaluate the evolution and harmony of the color palette across stages. Two metrics are computed:

- **Mean Hue**: dominant tonal direction (0 = red, 30 = yellow, 60 = green, 90 = cyan, 120 = blue, 150 = magenta)
- **Hue SD**: spread of colors; a proxy for palette harmony
  - Low Hue SD = harmonious, unified palette
  - High Hue SD = diverse, contrasting palette

In [ ]:
import cv2
import numpy as np

def hue_stats(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    hue = hsv[:, :, 0].flatten()
    return float(np.mean(hue)), float(np.std(hue))

mean_hues = []
std_hues = []

for i, img in enumerate(images):
    mean, std = hue_stats(img)
    mean_hues.append(mean)
    std_hues.append(std)
    print(f"Stage {i+1}: Mean Hue = {mean:.2f} | Hue SD = {std:.2f}")

### **3.6 Spatial Painting Density** — Edge Heatmaps

Each image is divided into a 20×30 grid. The density of edge pixels within each cell is measured and visualized as a heatmap, revealing where painting activity is spatially concentrated.

- **Localized hotspots**: areas of focused detail or shading effort
- **Expanding hot zones**: progressive work across new areas
- **Uniform density**: evenly distributed effort
- **Sparse zones**: backgrounds, negative space, untouched areas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import files

def edge_density_grid(image, grid_size=(20, 30)):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    h, w = edges.shape
    gh, gw = grid_size
    dh, dw = h // gh, w // gw
    heatmap = np.zeros((gh, gw))
    for i in range(gh):
        for j in range(gw):
            cell = edges[i*dh:(i+1)*dh, j*dw:(j+1)*dw]
            heatmap[i, j] = np.sum(cell > 0)
    return heatmap

all_heatmaps = [edge_density_grid(img, grid_size=(20, 30)) for img in images]
max_val = max(h.max() for h in all_heatmaps)

# Create output folder
output_dir = "heatmaps"
os.makedirs(output_dir, exist_ok=True)

for i, heatmap in enumerate(all_heatmaps):
    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(heatmap,
                cmap='hot',
                square=False,
                cbar=True,
                vmin=0,
                vmax=max_val,
                xticklabels=False,
                yticklabels=False,
                ax=ax)
    ax.set_title(f"Stage {i+1} — Spatial Painting Density")
    plt.tight_layout()

    filepath = os.path.join(output_dir, f"heatmap_stage_{i+1:02d}.png")
    plt.savefig(filepath, dpi=150)
    plt.show()
    plt.close()

# Zip and download
!zip -r heatmaps.zip heatmaps
files.download("heatmaps.zip")
print(f"Downloaded {len(all_heatmaps)} heatmaps as 'heatmaps.zip'")

---
## **4. Metrics Summary**

All extracted metrics are compiled into a single table, with one row per stage.

In [ ]:
import pandas as pd

image_metrics = pd.DataFrame({
    "Stage":                  [f"S{i}" for i in range(1, len(images) + 1)],
    "Stroke_Density_Pct":     stroke_densities_norm,
    "Texture_LBP_SD":         texture_scores,
    "Fractal_Dimension":      fd_scores,
    "Mean_Hue":               mean_hues,
    "Hue_SD":                 std_hues
})

print(f"Stages: {len(image_metrics)}")
image_metrics

---
## **5. Metric Evolution Plots**

Each metric is plotted across stages to visualize how it evolves throughout the painting process.

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = [
    ("Stroke_Density_Pct",  "Stroke Density (%)"),
    ("Texture_LBP_SD",      "Texture LBP SD"),
    ("Fractal_Dimension",   "Fractal Dimension"),
    ("Mean_Hue",            "Mean Hue"),
    ("Hue_SD",              "Hue SD")
]

n = len(metrics_to_plot)
fig, axes = plt.subplots(n, 1, figsize=(7, n * 3))

for ax, (col, label) in zip(axes, metrics_to_plot):
    ax.plot(image_metrics["Stage"], image_metrics[col],
            marker='o', color='steelblue', linewidth=1.5)
    ax.set_title(label)
    ax.set_xlabel("Stage")
    ax.set_ylabel(label)
    ax.set_xticks(image_metrics["Stage"])
    ax.grid(False)

plt.suptitle("Metric Evolution Across Painting Stages", fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

---
## **6. Export**

The metrics table is saved as a CSV file and downloaded locally. This file serves as the input for the statistical analysis in R.

In [ ]:
from google.colab import files

output_filename = f"{painting_id}_image.csv"
image_metrics.to_csv(output_filename, index=False)
files.download(output_filename)

print(f"Saved: {output_filename}")

---
*Both in One — Julieta Arco*